In [1]:
# --- CELL 1: DIAGNOSTIC SCRIPT ---
import torch
import sys
import os
from pathlib import Path
import pandas as pd
from torch.utils.data import DataLoader
import pandas as pd
from rdkit import Chem

In [2]:
# 1. SETUP (Adjust paths if necessary)
cwd = Path.cwd()
cwd = cwd.parents[0]

In [3]:
# Assuming your notebook is in the root or similar structure
sys.path.insert(0, os.path.join(cwd, 'data_loaders', 'AlignUniform'))
sys.path.insert(0, os.path.join(cwd, 'model'))
sys.path.insert(0, os.path.join(os.path.dirname(cwd.parent), 'tmach007/massformer/src/massformer'))
print(sys.path)

['/data/nas-gpu/wang/tmach007/massformer/src/massformer', '/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/model', '/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/data_loaders/AlignUniform', '/data/nas-gpu/wang/tmach007/massformer/y/envs/MF-GPU/lib/python38.zip', '/data/nas-gpu/wang/tmach007/massformer/y/envs/MF-GPU/lib/python3.8', '/data/nas-gpu/wang/tmach007/massformer/y/envs/MF-GPU/lib/python3.8/lib-dynload', '', '/data/nas-gpu/wang/tmach007/massformer/y/envs/MF-GPU/lib/python3.8/site-packages']


In [4]:
# Import your CURRENT loader (the one giving 0.0 validation)
try:
    from finetune_dataloader_au import AlignUniformDataset, au_collate_fn
except ImportError:
    print("⚠️ Could not import AlignUniformDataset. Make sure finetune_dataloader_au.py is in the folder.")

In [5]:
# 2. CONFIGURATION
class CheckArgs:
    pairs_path = "/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/data_splits/stratified_binary_07_dataset_val.feather"
    spec_data_path = "/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/mass_spec_gym_data/spec_df.pkl"
    mol_data_path = "/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/mass_spec_gym_data/mol_df.pkl"

args = CheckArgs()

In [6]:
# 3. EXECUTION
def check_identity(args):
    print("--- 1. Loading Dataset (First 500ms) ---")
    ds = AlignUniformDataset(args.pairs_path, args.spec_data_path, args.mol_data_path)
    
    print("\n--- 2. Checking Graph Identity (Item 0) ---")
    # Fetch the pair
    graph_A, graph_B = ds[0]
    
    # A. Check Node Features (Atoms)
    # x is [Num_Atoms, Num_Features]
    if graph_A.x.shape != graph_B.x.shape:
        print("✅ Shapes differ! Graphs are distinct.")
        return

    nodes_equal = torch.equal(graph_A.x, graph_B.x)
    print(f"► Node Features Identical?  {nodes_equal}")
    
    if not nodes_equal:
        diff = (graph_A.x - graph_B.x).abs().sum().item()
        print(f"   (Difference magnitude: {diff:.4f})")

    # B. Check Edge Indices (Connections)
    # edge_index is [2, Num_Edges]
    edges_equal = torch.equal(graph_A.edge_index, graph_B.edge_index)
    print(f"► Edge Structure Identical? {edges_equal}")
    
    # C. Verdict
    if nodes_equal and edges_equal:
        print("\n🚨 VERDICT: GRAPHS ARE CLONES.")
        print("   This explains why Validation Alignment is 0.0000.")
    else:
        print("\n✅ VERDICT: Graphs are distinct.")

In [7]:
check_identity(args)

--- 1. Loading Dataset (First 500ms) ---
Loading pairs from /data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/data_splits/stratified_binary_07_dataset_val.feather...
Filtered for Positive Pairs: 2478

--- 2. Checking Graph Identity (Item 0) ---
✅ Shapes differ! Graphs are distinct.


In [8]:
# 1. Load one pair
ds = AlignUniformDataset(args.pairs_path, args.spec_data_path, args.mol_data_path)
graph_A, graph_B = ds[0]

Loading pairs from /data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/data_splits/stratified_binary_07_dataset_val.feather...
Filtered for Positive Pairs: 2478


In [10]:
# Check Edge Connectivity (Edge Index)
# (Requires same number of edges to compare directly)
if graph_A.edge_index.shape == graph_B.edge_index.shape:
    diff_edges = (graph_A.edge_index - graph_B.edge_index).abs().sum().item()
else:
    diff_edges = "Shapes differ (Definitely distinct!)"

In [75]:
print(f"--- DIAGNOSIS ---")
print(f"Difference in Node Values: {diff_nodes:.6f}")
print(f"Difference in Edge Indices: {diff_edges}")

--- DIAGNOSIS ---
Difference in Node Values: 0.000000
Difference in Edge Indices: 0


In [76]:
if diff_nodes == 0.0 and diff_edges == 0:
    print("\n🚨 CONCLUSION: The graphs are value-identical clones.")
    print("   This confirms RDKit is canonicalizing them to the exact same atom order.")
else:
    print("\n✅ CONCLUSION: The graphs are distinct views.")


🚨 CONCLUSION: The graphs are value-identical clones.
   This confirms RDKit is canonicalizing them to the exact same atom order.


In [11]:
# --- 2. CONFIGURATION ---
class InspectArgs:
    # Update these paths to match your actual files
    pairs_path = "/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/data_splits/stratified_binary_07_dataset_val.feather" # Using VAL set
    spec_data_path = "/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/mass_spec_gym_data/spec_df.pkl"
    mol_data_path = "/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/mass_spec_gym_data/mol_df.pkl"

args = InspectArgs()

In [12]:
# --- 3. INSPECTION LOGIC ---
def inspect_validation_pairs(args, num_samples=100):
    print(f"--- 🔍 INSPECTING {num_samples} VALIDATION PAIRS ---")
    
    # Initialize Dataset
    ds = AlignUniformDataset(args.pairs_path, args.spec_data_path, args.mol_data_path)
    print(f"Total Validation Pairs: {len(ds)}")
    
    clone_count = 0
    distinct_count = 0
    
    print(f"\n{'IDX':<5} | {'STATUS':<10} | {'MOL_ID':<15} | {'SPEC_IDs (A vs B)':<30} | {'NODE DIFF':<10}")
    print("-" * 90)
    
    # Iterate
    for i in range(min(len(ds), num_samples)):
        try:
            # 1. Get Graphs
            graph_A, graph_B = ds[i]
            
            # 2. Get Metadata (to see what molecule this is)
            pair_info = ds.pos_pairs.iloc[i]
            spec_id_A = pair_info['name_main']
            spec_id_B = pair_info['name_sub']
            
            # Lookup Mol ID
            mol_id = ds.spec_lookup.loc[spec_id_A, 'mol_id']
            
            # 3. Check for Identity (The "Clone" Test)
            nodes_equal = torch.equal(graph_A.x, graph_B.x)
            edges_equal = torch.equal(graph_A.edge_index, graph_B.edge_index)
            
            is_clone = nodes_equal and edges_equal
            
            # 4. Calculate Difference Magnitude
            diff = 0.0
            if not nodes_equal:
                diff = (graph_A.x - graph_B.x).abs().sum().item()
            
            # 5. Log Result
            status = "🔴 CLONE" if is_clone else "✅ DISTINCT"
            if is_clone: clone_count += 1
            else: distinct_count += 1
            
            # Print row
            print(f"{i:<5} | {status:<10} | {str(mol_id)[:15]:<15} | {str(spec_id_A)[:20]} vs {str(spec_id_B)[:20]} | {diff:.4f}")

            # Optional: Print SMILES for the first clone found
            if is_clone and clone_count == 1:
                smiles = ds.mol_lookup.loc[mol_id, 'smiles'] # Assuming column is 'smiles'
                print(f"   [!] Example Clone Structure (SMILES): {smiles}")

        except Exception as e:
            print(f"{i:<5} | ⚠️ ERROR    | {str(e)}")

    print("-" * 90)
    print(f"\n--- SUMMARY ---")
    print(f"Total Analyzed: {clone_count + distinct_count}")
    print(f"Identical Clones: {clone_count} ({(clone_count / (clone_count + distinct_count))*100:.1f}%)")
    print(f"Distinct Views:   {distinct_count}")
    
    if clone_count > 0:
        print("\n🚨 DIAGNOSIS: The dataset contains chemically identical graphs.")
        print("   Reason: RDKit canonicalizes the atom order, so 'Graph A' and 'Graph B'")
        print("   are mathematically equal matrices, leading to 0.0 distance.")

In [13]:
# --- 4. RUN ---
inspect_validation_pairs(args)

--- 🔍 INSPECTING 100 VALIDATION PAIRS ---
Loading pairs from /data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/data_splits/stratified_binary_07_dataset_val.feather...
Filtered for Positive Pairs: 2478
Total Validation Pairs: 2478

IDX   | STATUS     | MOL_ID          | SPEC_IDs (A vs B)              | NODE DIFF 
------------------------------------------------------------------------------------------
0     | ⚠️ ERROR    | The size of tensor a (16) must match the size of tensor b (21) at non-singleton dimension 0
1     | ⚠️ ERROR    | The size of tensor a (30) must match the size of tensor b (29) at non-singleton dimension 0
2     | ⚠️ ERROR    | The size of tensor a (19) must match the size of tensor b (36) at non-singleton dimension 0
3     | ✅ DISTINCT | 18314           | MassSpecGymID0216000 vs MassSpecGymID0216424 | 145.0000
4     | ⚠️ ERROR    | The size of tensor a (43) must match the size of tensor b (42) at non-singleton dimension 0
5     | ⚠️ ERROR    | The size of tens

## Are molecules chemically distinct

In [58]:
import torch
import pandas as pd
import numpy as np
import sys
import os
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs

In [59]:
# --- 1. SETUP ---
cwd = Path.cwd()
cwd = cwd.parents[0]

In [60]:
sys.path.insert(0, os.path.join(cwd, 'data_loaders', 'AlignUniform'))
sys.path.insert(0, os.path.join(cwd, 'model'))
sys.path.insert(0, os.path.join(os.path.dirname(cwd.parent), 'tmach007/massformer/src/massformer'))

In [61]:
try:
    from finetune_dataloader_au import AlignUniformDataset
except ImportError:
    # If import fails, we define a minimal class to load data
    print("⚠️ Could not import loader. Please ensure finetune_dataloader_au.py is in the folder.")

In [62]:
# --- 2. CONFIGURATION ---
class InspectArgs:
    # Update these paths to your VALIDATION set
    pairs_path = "/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/data_splits/stratified_binary_07_dataset_val.feather"
    spec_data_path = "/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/mass_spec_gym_data/spec_df_COMBINED.pkl"
    mol_data_path = "/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/mass_spec_gym_data/mol_df_COMBINED.pkl"

args = InspectArgs()

In [63]:
# --- 3. INSPECTION FUNCTION ---
def inspect_chemical_vs_graph(args, num_samples=10):
    print(f"--- 🧪 ANALYZING {num_samples} PAIRS: CHEMICAL vs GRAPH IDENTITY ---")
    
    ds = AlignUniformDataset(args.pairs_path, args.spec_data_path, args.mol_data_path)
    
    print(f"{'IDX':<4} | {'TYPE':<10} | {'SMILES COMPARISON':<20} | {'TANIMOTO':<8} | {'GRAPH DIFF':<10}")
    print("-" * 80)
    
    for i in range(num_samples):
        try:
            # 1. Get Graphs (Tensor World)
            graph_A, graph_B = ds[i]
            
            # 2. Get Metadata (Chemical World)
            pair_row = ds.pos_pairs.iloc[i]
            spec_A_id = pair_row['name_main']
            spec_B_id = pair_row['name_sub']
            
            mol_id_A = ds.spec_lookup.loc[spec_A_id, 'mol_id']
            # We assume label=1, but let's check what B actually is
            mol_id_B = ds.spec_lookup.loc[spec_B_id, 'mol_id']

            # 3. Get SMILES & Tanimoto
            # Generate RDKit Mols
            mol_obj_A = ds.mol_lookup.loc[mol_id_A, 'mol']
            mol_obj_B = ds.mol_lookup.loc[mol_id_B, 'mol'] # Should be same for Label=1
            
            smiles_A = Chem.MolToSmiles(mol_obj_A)
            smiles_B = Chem.MolToSmiles(mol_obj_B)
            
            # Calculate Tanimoto (Chemical Similarity)
            fp_A = AllChem.GetMorganFingerprintAsBitVect(mol_obj_A, 2, nBits=1024)
            fp_B = AllChem.GetMorganFingerprintAsBitVect(mol_obj_B, 2, nBits=1024)
            tanimoto = DataStructs.TanimotoSimilarity(fp_A, fp_B)
            
            # 4. Calculate Graph Difference (Tensor L1 Norm)
            if graph_A.x.shape == graph_B.x.shape:
                diff_graph = (graph_A.x - graph_B.x).abs().sum().item()
            else:
                diff_graph = -1.0 # Shape mismatch
            
            # 5. Determine Status
            chem_status = "SAME" if smiles_A == smiles_B else "DIFF"
            
            print(f"{i:<4} | {chem_status:<10} | {smiles_A[:10]}... vs {smiles_B[:10]}... | {tanimoto:.4f}   | {diff_graph:.4f}")
            
            if i == 0:
                print(f"     Full SMILES A: {smiles_A}")
                if smiles_A != smiles_B:
                     print(f"     Full SMILES B: {smiles_B}")

        except Exception as e:
            print(f"{i:<4} | ERROR      | {str(e)}")

    print("-" * 80)
    print("\n--- DIAGNOSIS GUIDE ---")
    print("1. If SMILES are SAME and Graph Diff is 0.0000:")
    print("   -> You are pairing identical molecules. RDKit canonicalization makes them clones.")
    print("   -> FIX: Use 'AugmentedAlignUniformDataset' (Atom Shuffling).")
    print("\n2. If SMILES are DIFF and Graph Diff is 0.0000:")
    print("   -> Highly unlikely. Means different molecules map to identical graphs (Collision).")
    print("\n3. If SMILES are SAME and Graph Diff > 0:")
    print("   -> Perfect! Your augmentation is working. The model sees different views of the same thing.")

In [64]:
inspect_chemical_vs_graph(args, num_samples=10)

--- 🧪 ANALYZING 10 PAIRS: CHEMICAL vs GRAPH IDENTITY ---
Loading pairs from /data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/data_splits/stratified_binary_07_dataset_val.feather...
Filtered for Positive Pairs: 2478
Loading spectral data...
Loading molecular data...
IDX  | TYPE       | SMILES COMPARISON    | TANIMOTO | GRAPH DIFF
--------------------------------------------------------------------------------
0    | DIFF       | Nc1ccn(C2C... vs Nc1ccn(C2O... | 0.5962   | 0.0000
     Full SMILES A: Nc1ccn(C2CC(O)C(CO)O2)c(=O)n1
     Full SMILES B: Nc1ccn(C2OC(CO)C(OP(=O)(O)O)C2O)c(=O)n1
1    | DIFF       | COc1ccc(S(... vs Cc1ccc(S(=... | 0.8136   | 0.0000
2    | DIFF       | C#CC1(O)C(... vs CCCCCCCCC=... | 0.3944   | 0.0000
3    | DIFF       | COc1ccc(S(... vs CCc1ccc(S(... | 0.7869   | 0.0000
4    | DIFF       | CC1OC(OCC2... vs O=c1c(OC2O... | 0.7910   | 0.0000
5    | DIFF       | COC1C(O)C(... vs Nc1ccn(C2O... | 0.6604   | 0.0000
6    | DIFF       | N#CC1C(O)C... vs CC1(F)C(

## Check if the molecules paired up are different

In [85]:
def verify_molecular_identity(args, num_checks=5):
    print(f"--- 🧬 DNA TEST: Are these actually the same molecule? ---")
    
    # Load Data
    ds = AlignUniformDataset(args.pairs_path, args.spec_data_path, args.mol_data_path)
    
    for i in range(num_checks):
        print(f"\n[PAIR {i}]")
        
        # 1. Get the IDs from the Pair List
        pair_row = ds.pos_pairs.iloc[i]
        spec_id_A = pair_row['name_main']
        spec_id_B = pair_row['name_sub']
        
        # 2. Retrieve the Molecule Objects
        mol_id_A = ds.spec_lookup.loc[spec_id_A, 'mol_id']
        mol_id_B = ds.spec_lookup.loc[spec_id_B, 'mol_id']
        
        mol_obj_A = ds.mol_lookup.loc[mol_id_A, 'mol']
        mol_obj_B = ds.mol_lookup.loc[mol_id_B, 'mol']
        
        # 3. GENERATE THE KEYS (The Proof)
        key_A = Chem.MolToInchiKey(mol_obj_A)
        key_B = Chem.MolToInchiKey(mol_obj_B)
        
        print(f"   Spec A ({spec_id_A}): {key_A}")
        print(f"   Spec B ({spec_id_B}): {key_B}")
        
        # 4. VERDICT
        if key_A == key_B:
            print("   ✅ VERDICT: CONFIRMED. Same Molecule (Rex).")
            print("      (Different spectra, same chemical structure).")
        else:
            print("   ❌ VERDICT: ERROR. Different Molecules!")
            print("      (The dataset labeled these as a pair, but they are chemically distinct).")

In [86]:
# Run checking
verify_molecular_identity(args)

--- 🧬 DNA TEST: Are these actually the same molecule? ---
Loading pairs from /data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/data_splits/stratified_binary_07_dataset_val.feather...
Filtered for Positive Pairs: 2478
Loading spectral data...
Loading molecular data...

[PAIR 0]
   Spec A (MassSpecGymID0128595): CKTSBUTUHBMZGZ-UHFFFAOYSA-N
   Spec B (MassSpecGymID0176777): UOOOPKANIPLQPU-UHFFFAOYSA-N
   ❌ VERDICT: ERROR. Different Molecules!
      (The dataset labeled these as a pair, but they are chemically distinct).

[PAIR 1]
   Spec A (MassSpecGymID0216002): NHWURLWKNOOTLU-UHFFFAOYSA-N
   Spec B (MassSpecGymID0216178): XGJHIUYQKVQASN-UHFFFAOYSA-N
   ❌ VERDICT: ERROR. Different Molecules!
      (The dataset labeled these as a pair, but they are chemically distinct).

[PAIR 2]
   Spec A (MassSpecGymID0218985): JFIWEPHGRUDAJN-UHFFFAOYSA-N
   Spec B (MassSpecGymID0228457): FLFGNMFWNBOBGE-UHFFFAOYSA-N
   ❌ VERDICT: ERROR. Different Molecules!
      (The dataset labeled these as a pa